# 02 — Character and word tokenizers (Part B)

First the 🧩 building blocks (vocabularies, encoding, counting), then you package them in `src/char_tokenizer.py` and `src/word_tokenizer.py` (`pytest tests/test_tokenizers.py`), then you run the experiments on a **real** corpus (Tiny Shakespeare).

**How the 🧩 exercises work.** Cells marked 🧩 contain `____` blanks: complete them using the hints (they name the Python functions you need). Each exercise is followed by a ✅ *check* cell that tells you immediately whether you got it right. When an exercise matches a function of `src/`, the docstring of that function says so: you then **move your working code into `src/`** and run `pytest`.

In [ ]:
import sys, pathlib, json, time
ROOT = pathlib.Path.cwd().parent          # notebooks/ is one level below the repo root
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
%load_ext autoreload
%autoreload 2

from collections import Counter
from src.char_tokenizer import CharacterTokenizer
from src.word_tokenizer import WordTokenizer, word_tokenize     # word_tokenize is PROVIDED
from src.preprocessing import preprocess_text
from src.evaluation import split_lines

## Task 2 — building blocks of a character tokenizer

### 🧩 Exercise 2.1 — A character vocabulary with special tokens

Build `token_to_id` and `id_to_token` for the text below: the four special tokens come first (ids 0–3), then every distinct character in **sorted** order.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `set(text)` gives the distinct characters; `sorted(...)` makes the order reproducible.
* Concatenate lists with `+`: `SPECIALS + chars`.
* `enumerate(items)` yields `(0, item0), (1, item1), …`.
* Inverting a dict: `{v: k for k, v in d.items()}`.

In [ ]:
SPECIALS = ["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
text = "abracadabra, a cat!"

chars = ____(____)
token_to_id = {tok: i for i, tok in enumerate(____)}
id_to_token = {____}
print(token_to_id)

In [ ]:
assert token_to_id["[PAD]"] == 0 and token_to_id["[EOS]"] == 3
assert [id_to_token[i] for i in range(4, 8)] == [" ", "!", ",", "a"]
assert len(token_to_id) == 4 + len(set(text))
print("✅ Exercise 2.1 OK")

### 🧩 Exercise 2.2 — Encode and decode

Write `encode` (unknown characters → the id of `[UNK]`, optional `[BOS]`/`[EOS]`) and `decode` (drop the special tokens).

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `dict.get(key, default)` returns `default` when the key is missing — perfect for unknown characters.
* A list comprehension builds the id list in one line.
* To wrap: `[bos_id] + ids + [eos_id]`.
* In `decode`, skip ids whose token is in `SPECIALS`.

In [ ]:
unk_id = token_to_id["[UNK]"]

def encode(text, add_special_tokens=False):
    ids = [token_to_id.____(ch, ____) for ch in text]
    if add_special_tokens:
        ids = [token_to_id["[BOS]"]] + ____ + [token_to_id["[EOS]"]]
    return ids

def decode(ids):
    return "".join(id_to_token[i] for i in ids if ____)

print(encode("cat!?"), decode(encode("cat", add_special_tokens=True)))

In [ ]:
assert encode("cat") == [token_to_id[c] for c in "cat"]
assert encode("z")[0] == unk_id                       # 'z' is not in the vocabulary
assert encode("a", add_special_tokens=True) == [2, token_to_id["a"], 3]
assert decode(encode("a cat", add_special_tokens=True)) == "a cat"
print("✅ Exercise 2.2 OK")

## Task 3 — building blocks of a word tokenizer

### 🧩 Exercise 2.3 — Count, filter and order the words

Count the tokens of a small corpus, keep those seen at least `min_freq` times, and order them by decreasing count (ties: alphabetical).

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* `Counter(iterable)` counts how many times each item occurs; `counter.items()` yields `(item, count)`.
* Filter with a list comprehension and a condition on the count.
* `list.sort(key=...)`: use a **tuple** as the key; to sort a number in decreasing order, negate it. Ties then fall through to the second element of the tuple.
* `word_tokenize(text)` (provided) splits text into word and punctuation tokens.

In [ ]:
corpus = ["the cat sat on the mat .", "the dog sat ."]
min_freq = 2

counts = ____(tok for line in corpus for tok in word_tokenize(line))
kept = [(tok, c) for tok, c in counts.items() if c ____]
kept.sort(key=lambda tc: (____, ____))
print(kept)

In [ ]:
assert kept == [("the", 3), (".", 2), ("sat", 2)]
print("✅ Exercise 2.3 OK")

### 🧩 Exercise 2.4 — Out-of-vocabulary rate

Given the list of ids of a text and the id of `[UNK]`, compute the fraction of **occurrences** (not distinct words) that are unknown.

Replace every `____` with the right code, run the cell, then run the **check** cell below it.

**Hints**
* Count with `sum(1 for i in ids if i == unk_id)`.
* Divide by `len(ids)`; return `0.0` if the list is empty.

In [ ]:
def oov_rate(ids, unk_id):
    if not ids:
        return 0.0
    return ____

print(oov_rate([5, 1, 7, 1, 1], unk_id=1))

In [ ]:
assert oov_rate([5, 1, 7, 1, 1], 1) == 0.6 and oov_rate([], 1) == 0.0
print("✅ Exercise 2.4 OK")

## ⏸ Stop: package your code

| Exercise | → `src/` |
|---|---|
| 2.1 | `CharacterTokenizer.train` |
| 2.2 | `CharacterTokenizer.encode` / `decode` (and the same logic for `WordTokenizer`) |
| 2.3 | `WordTokenizer.train` |
| 2.4 | `WordTokenizer.oov_rate` |

Run `pytest tests/test_tokenizers.py` and continue when it is green.

## Load and split the real dataset

In [ ]:
raw = (ROOT / "data/real/tinyshakespeare.txt").read_text(encoding="utf-8").splitlines()
lines = [l for l in (preprocess_text(x) for x in raw) if l]     # we do NOT deduplicate: repeated speaker names are real data
train_lines, test_lines = split_lines(lines, test_fraction=0.2, seed=0)
print(len(lines), "lines |", len(train_lines), "train |", len(test_lines), "test"); print(train_lines[:3])

In [ ]:
ctok = CharacterTokenizer(); ctok.train(train_lines)
sample = "To be, or not to be: that is the question."
ids = ctok.encode(sample, add_special_tokens=True)
print("vocab size:", ctok.vocab_size); print(ids); print(repr(ctok.decode(ids))); print("unknown char:", ctok.encode("日本"))

**Questions (character level)**
1. Main advantage? 2. Main limitation? 3. Why are the sequences long? (use the numbers) 4. How does the vocabulary compare with word level?

**Your answer:**

_(write here)_

In [ ]:
wtok = WordTokenizer(); wtok.train(train_lines)
print("vocab size:", wtok.vocab_size)
s = "Whither thou goest, I will follow thee."
print(word_tokenize(s)); print(wtok.encode(s)); print(repr(wtok.decode(wtok.encode(s))))
print("OOV rate on TEST lines:", round(wtok.oov_rate(test_lines), 4))

### The OOV / vocabulary-size trade-off

In [ ]:
rows = []
for min_freq in [1, 2, 3, 5, 10, 20]:
    t = WordTokenizer(min_freq=min_freq); t.train(train_lines)
    rows.append({"min_freq": min_freq, "vocab_size": t.vocab_size, "oov_train": t.oov_rate(train_lines), "oov_test": t.oov_rate(test_lines)})
sweep = pd.DataFrame(rows); sweep

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(sweep.min_freq, sweep.vocab_size, marker="o"); ax[0].set_xlabel("min_freq"); ax[0].set_ylabel("vocab size"); ax[0].set_title("Vocabulary size")
ax[1].plot(sweep.min_freq, sweep.oov_test, marker="o", label="test"); ax[1].plot(sweep.min_freq, sweep.oov_train, marker="o", label="train")
ax[1].set_xlabel("min_freq"); ax[1].set_ylabel("OOV rate"); ax[1].legend(); ax[1].set_title("OOV rate")
plt.tight_layout(); plt.savefig(ROOT / "results/figures/word_oov_sweep.png", dpi=150); plt.show()

In [ ]:
n_ctok = sum(len(ctok.encode(l)) for l in test_lines); n_words = sum(len(word_tokenize(l)) for l in test_lines)
pd.DataFrame({"tokenizer": ["character", "word (min_freq=1)"], "vocab_size": [ctok.vocab_size, wtok.vocab_size],
              "tokens_on_test": [n_ctok, n_words], "avg_tokens_per_line": [n_ctok / len(test_lines), n_words / len(test_lines)]})

**Questions (word level)**
1. What happens when the tokenizer meets an unknown word? Show an example from `test_lines`.
2. What is the OOV problem? Read it off your plot.
3. Why can a word vocabulary become huge? 4. Why is word-level tokenization insufficient for modern LLMs?
5. Is `decode(encode(x)) == x` always true? Find a counter-example in the data.

**Your answer:**

_(write here)_